In [ ]:
"""
01_psf_extract_from_tiff.py

PSF estimation from widefield fluorescent bead images (many beads per frame)
Instrument: Nikon Ti2 widefield, 100x oil objective, NA 1.49

What this script does
- Detect beads using LoG blob detection (skimage.feature.blob_log)
- Reject aggregates/overlaps/saturated/low-SNR/edge beads
- Fit each accepted bead with 2D Gaussian + constant background
- Output FWHM_x, FWHM_y (nm) for each bead + summary stats
- Produce QC images showing which beads were accepted/rejected and why

Outputs (created in: <INPUT_FOLDER>/PSF_analysis_outputs/)
- <image>_psf_results.csv           (accepted beads + fit params + FWHM)
- <image>_psf_rejected.csv          (rejected candidates + reasons)
- <image>_psf_summary.txt           (per-image summary)
- <image>_qc_overlay.png            (accepted circled; rejected X + reason)
- <image>_qc_rois_accepted.png      (montage of accepted ROIs)
- ALL_psf_results.csv               (combined accepted across files)
- ALL_psf_rejected.csv              (combined rejected across files)
- ALL_psf_summary.txt               (combined summary)

Dependencies:
  pip install numpy scipy pandas matplotlib scikit-image tifffile

IMPORTANT:
- Set PIXEL_SIZE_UM correctly (um/px at the sample plane).
"""

from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------- robust image readers --------
try:
    from tifffile import imread as tiff_imread
    HAVE_TIFFFILE = True
except Exception:
    HAVE_TIFFFILE = False
    from skimage.io import imread as sk_imread  # fallback

from skimage.feature import blob_log
from skimage.morphology import disk
from skimage.filters import median
from scipy.optimize import curve_fit


# =============================================================================
# USER SETTINGS (edit these)
# =============================================================================

# Folder containing your TIFF images (use a relative path for GitHub friendliness)
INPUT_FOLDER = Path("data/example_psf")  # e.g., Path("data/example_psf") or Path("/path/to/folder")

# Choose ONE of the following:
FILE_NAME = None          # e.g., "5_FL.tif" (single file). If not None, FILE_GLOB is ignored.
FILE_GLOB = "*.tif"       # e.g., "*.tif" for batch processing. Set to None if using FILE_NAME.

# Pixel size at sample plane (um/px). MUST be correct for your microscope/camera/binning.
PIXEL_SIZE_UM = 0.065     # example: 0.065 um/px = 65 nm/px (replace with your calibrated value)

# Camera saturation level (depends on bit depth)
SATURATION_LEVEL = 65535  # 16-bit typical; 4095 for 12-bit; 16383 for 14-bit

# Bead info
BEAD_DIAMETER_NM = 200.0
DO_BEAD_SIZE_CORRECTION = True  # quadrature subtraction (approx)

# =============================================================================
# DETECTION / FILTER PARAMETERS
# =============================================================================
USE_MEDIAN_FILTER = True
MEDIAN_RADIUS_PX = 1

# LoG blob detection settings
LOG_MIN_SIGMA = 0.8
LOG_MAX_SIGMA = 3.0
LOG_NUM_SIGMA = 10
LOG_THRESHOLD = 0.02      # higher => fewer beads detected

# ROI settings
ROI_HALF_SIZE_PX = 10     # ROI size = (2*half+1)
BG_INNER_R_PX = 12
BG_OUTER_R_PX = 18

# Filtering criteria
MIN_PEAK_SNR = 8.0
MIN_NEIGHBOR_DIST_PX = 2 * ROI_HALF_SIZE_PX
MAX_ELLIPTICITY = 2.0
MAX_CENTER_SHIFT_PX = 2.0


# =============================================================================
# MODEL + HELPERS
# =============================================================================
def gaussian2d(coords, amp, x0, y0, sx, sy, offset):
    (x, y) = coords
    g = amp * np.exp(
        -(((x - x0) ** 2) / (2 * sx**2) + ((y - y0) ** 2) / (2 * sy**2))
    ) + offset
    return g.ravel()


def fwhm_from_sigma(sigma_px: float, pixel_size_um: float) -> float:
    """FWHM (nm) from Gaussian sigma in pixels."""
    return 2.355 * sigma_px * pixel_size_um * 1000.0


def bead_sigma_nm(bead_diameter_nm: float) -> float:
    """Approximate disk->Gaussian sigma heuristic (used only for approximate correction)."""
    r = bead_diameter_nm / 2.0
    return r / np.sqrt(2.0)


def quadrature_subtract_nm(measured_nm: float, subtract_nm: float) -> float:
    v2 = measured_nm**2 - subtract_nm**2
    return float(np.sqrt(v2)) if v2 > 0 else np.nan


def load_image(path: Path) -> np.ndarray:
    if HAVE_TIFFFILE:
        img = tiff_imread(str(path))
    else:
        img = sk_imread(str(path))

    # If multi-dimensional, take the first frame by default
    if img.ndim > 2:
        img = img[0]
    return img.astype(np.float64)


def annulus_mask(h: int, w: int, cx: float, cy: float, r_in: float, r_out: float) -> np.ndarray:
    yy, xx = np.ogrid[:h, :w]
    rr = (xx - cx) ** 2 + (yy - cy) ** 2
    return (rr >= r_in**2) & (rr <= r_out**2)


def extract_roi(img: np.ndarray, x: float, y: float, half: int) -> np.ndarray | None:
    h, w = img.shape
    x = int(round(x))
    y = int(round(y))
    x0, x1 = x - half, x + half + 1
    y0, y1 = y - half, y + half + 1
    if x0 < 0 or y0 < 0 or x1 > w or y1 > h:
        return None
    return img[y0:y1, x0:x1]


def fit_bead_roi(roi: np.ndarray):
    h, w = roi.shape
    yy, xx = np.mgrid[0:h, 0:w]

    offset0 = np.percentile(roi, 10)
    amp0 = np.max(roi) - offset0
    x0 = (w - 1) / 2
    y0 = (h - 1) / 2
    sx0 = 1.3
    sy0 = 1.3

    p0 = [amp0, x0, y0, sx0, sy0, offset0]
    bounds = ([0, 0, 0, 0.3, 0.3, -np.inf], [np.inf, w - 1, h - 1, 10.0, 10.0, np.inf])

    try:
        popt, _ = curve_fit(gaussian2d, (xx, yy), roi.ravel(), p0=p0, bounds=bounds, maxfev=8000)
        fit = gaussian2d((xx, yy), *popt).reshape(h, w)
        resid = roi - fit
        rmse = float(np.sqrt(np.mean(resid**2)))
        return popt, rmse
    except Exception:
        return None, np.nan


def make_montage(rois: list[np.ndarray], ncols: int = 10, pad: int = 2) -> np.ndarray | None:
    if len(rois) == 0:
        return None
    rh, rw = rois[0].shape
    n = len(rois)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))

    canvas = np.zeros((nrows * (rh + pad) + pad, ncols * (rw + pad) + pad), dtype=np.float64)
    idx = 0
    for r in range(nrows):
        for c in range(ncols):
            if idx >= n:
                break
            y0 = pad + r * (rh + pad)
            x0 = pad + c * (rw + pad)
            canvas[y0 : y0 + rh, x0 : x0 + rw] = rois[idx]
            idx += 1
    return canvas


def robust_display_image(img: np.ndarray) -> np.ndarray:
    p1, p995 = np.percentile(img, [1, 99.5])
    if (not np.isfinite(p1)) or (not np.isfinite(p995)) or (p995 <= p1):
        p1, p995 = float(np.min(img)), float(np.max(img))
    return np.clip((img - p1) / (p995 - p1 + 1e-12), 0, 1)


# =============================================================================
# MAIN PSF LOGIC
# =============================================================================
def process_one_image(img: np.ndarray, pixel_size_um: float, out_dir: Path, base_name: str):
    out_dir.mkdir(parents=True, exist_ok=True)

    img_proc = img.copy()
    if USE_MEDIAN_FILTER:
        img_proc = median(img_proc, footprint=disk(MEDIAN_RADIUS_PX))

    # Normalize for detection
    img_norm = img_proc - np.percentile(img_proc, 1)
    img_norm = np.clip(img_norm, 0, None)
    m = np.max(img_norm)
    if m > 0:
        img_norm = img_norm / m

    blobs = blob_log(
        img_norm,
        min_sigma=LOG_MIN_SIGMA,
        max_sigma=LOG_MAX_SIGMA,
        num_sigma=LOG_NUM_SIGMA,
        threshold=LOG_THRESHOLD,
    )
    candidates = [(float(x), float(y), float(s)) for (y, x, s) in blobs]

    # Sort by center intensity (brightest first)
    peaks = []
    for x, y, s in candidates:
        xi, yi = int(round(x)), int(round(y))
        if 0 <= yi < img.shape[0] and 0 <= xi < img.shape[1]:
            peaks.append(img[yi, xi])
        else:
            peaks.append(-np.inf)
    order = np.argsort(peaks)[::-1]
    candidates = [candidates[i] for i in order]

    xy_all = np.array([(c[0], c[1]) for c in candidates], dtype=np.float64) if candidates else np.zeros((0, 2))
    accepted = []
    rejected = []
    taken = np.zeros(len(candidates), dtype=bool)

    for i, (x, y, s) in enumerate(candidates):
        if taken[i]:
            continue

        reasons = []
        roi = extract_roi(img, x, y, ROI_HALF_SIZE_PX)
        if roi is None:
            rejected.append((x, y, s, np.nan, np.nan, np.nan, np.nan, np.nan, "edge_roi_out"))
            continue

        peak = float(np.max(roi))
        if peak >= SATURATION_LEVEL:
            reasons.append("saturated")

        # Local background via annulus (median + robust std via MAD)
        h, w = img.shape
        mask = annulus_mask(h, w, x, y, BG_INNER_R_PX, BG_OUTER_R_PX)
        bg_vals = img[mask]

        if bg_vals.size < 50:
            bg_mean, bg_std = np.nan, np.nan
            reasons.append("bg_too_small")
        else:
            bg_mean = float(np.median(bg_vals))
            bg_std = float(np.median(np.abs(bg_vals - bg_mean)) * 1.4826 + 1e-12)

        snr = (peak - bg_mean) / bg_std if np.isfinite(bg_mean) else np.nan
        if (not np.isfinite(snr)) or (snr < MIN_PEAK_SNR):
            reasons.append("low_snr")

        # Neighbor distance to already accepted
        if accepted:
            acc_xy = np.array([(r["x_px"], r["y_px"]) for r in accepted], dtype=np.float64)
            dmin = float(np.min(np.sqrt((acc_xy[:, 0] - x) ** 2 + (acc_xy[:, 1] - y) ** 2)))
            if dmin < MIN_NEIGHBOR_DIST_PX:
                reasons.append("neighbor_too_close")

        if "neighbor_too_close" in reasons:
            rejected.append((x, y, s, peak, bg_mean, bg_std, snr, np.nan, " ; ".join(reasons)))
            continue

        roi_bs = np.clip(roi - (bg_mean if np.isfinite(bg_mean) else 0.0), 0, None)

        popt, rmse = fit_bead_roi(roi_bs)
        if popt is None:
            reasons.append("fit_failed")
            rejected.append((x, y, s, peak, bg_mean, bg_std, snr, np.nan, " ; ".join(reasons)))
            continue

        amp, x0, y0, sx, sy, offset = popt
        center_shift = float(
            np.sqrt((x0 - (roi_bs.shape[1] - 1) / 2) ** 2 + (y0 - (roi_bs.shape[0] - 1) / 2) ** 2)
        )
        ellip = float(max(sx, sy) / max(1e-12, min(sx, sy)))

        if center_shift > MAX_CENTER_SHIFT_PX:
            reasons.append("center_shift")
        if ellip > MAX_ELLIPTICITY:
            reasons.append("elliptical_aggregate")
        if rmse > (0.25 * np.max(roi_bs) + 1e-12):
            reasons.append("high_rmse")

        if not reasons:
            fwhm_x_nm = fwhm_from_sigma(float(sx), pixel_size_um)
            fwhm_y_nm = fwhm_from_sigma(float(sy), pixel_size_um)

            accepted.append(
                dict(
                    x_px=x,
                    y_px=y,
                    sigma_log_px=s,
                    peak=peak,
                    bg_mean=bg_mean,
                    bg_std=bg_std,
                    snr=snr,
                    amp=float(amp),
                    x0=float(x0),
                    y0=float(y0),
                    sx_px=float(sx),
                    sy_px=float(sy),
                    offset=float(offset),
                    rmse=float(rmse),
                    ellip=ellip,
                    center_shift_px=center_shift,
                    fwhm_x_nm=fwhm_x_nm,
                    fwhm_y_nm=fwhm_y_nm,
                )
            )

            # Suppress nearby candidates
            if len(xy_all) > 0:
                dist = np.sqrt((xy_all[:, 0] - x) ** 2 + (xy_all[:, 1] - y) ** 2)
                taken |= dist < MIN_NEIGHBOR_DIST_PX
        else:
            rejected.append((x, y, s, peak, bg_mean, bg_std, snr, rmse, " ; ".join(reasons)))

    df_acc = pd.DataFrame(accepted)
    df_rej = pd.DataFrame(
        rejected,
        columns=["x_px", "y_px", "sigma_log_px", "peak", "bg_mean", "bg_std", "snr", "rmse", "reasons"],
    )

    # Bead-size correction (approx)
    bead_sig_nm = bead_sigma_nm(BEAD_DIAMETER_NM) if DO_BEAD_SIZE_CORRECTION else 0.0
    bead_fwhm_nm = 2.355 * bead_sig_nm

    if DO_BEAD_SIZE_CORRECTION and len(df_acc) > 0:
        df_acc["fwhm_x_nm_corr"] = df_acc["fwhm_x_nm"].apply(lambda v: quadrature_subtract_nm(v, bead_fwhm_nm))
        df_acc["fwhm_y_nm_corr"] = df_acc["fwhm_y_nm"].apply(lambda v: quadrature_subtract_nm(v, bead_fwhm_nm))
    else:
        df_acc["fwhm_x_nm_corr"] = np.nan
        df_acc["fwhm_y_nm_corr"] = np.nan

    # Save tables
    df_acc.to_csv(out_dir / f"{base_name}_psf_results.csv", index=False)
    df_rej.to_csv(out_dir / f"{base_name}_psf_rejected.csv", index=False)

    # Summary text
    summary_lines = [
        f"File: {base_name}",
        f"Pixel size (um/px): {pixel_size_um}",
        f"Detected candidates: {len(candidates)}",
        f"Accepted beads: {len(df_acc)}",
        f"Rejected beads: {len(df_rej)}",
        "",
    ]

    if len(df_acc) > 0:
        fx = df_acc["fwhm_x_nm"].to_numpy()
        fy = df_acc["fwhm_y_nm"].to_numpy()
        summary_lines += [
            f"FWHM_x (nm): mean {np.nanmean(fx):.1f}, SD {np.nanstd(fx):.1f}",
            f"FWHM_y (nm): mean {np.nanmean(fy):.1f}, SD {np.nanstd(fy):.1f}",
        ]
        if DO_BEAD_SIZE_CORRECTION:
            fxc = df_acc["fwhm_x_nm_corr"].to_numpy()
            fyc = df_acc["fwhm_y_nm_corr"].to_numpy()
            summary_lines += [
                "",
                f"Bead correction ON (approx). Bead-equivalent FWHM ~ {bead_fwhm_nm:.1f} nm",
                f"Corrected FWHM_x (nm): mean {np.nanmean(fxc):.1f}, SD {np.nanstd(fxc):.1f}",
                f"Corrected FWHM_y (nm): mean {np.nanmean(fyc):.1f}, SD {np.nanstd(fyc):.1f}",
            ]
    else:
        summary_lines.append("No beads accepted. Tune LOG_THRESHOLD / MIN_PEAK_SNR / sigma range / ROI size.")

    (out_dir / f"{base_name}_psf_summary.txt").write_text("\n".join(summary_lines))

    # QC overlay
    try:
        disp = robust_display_image(img)
        fig, ax = plt.subplots(figsize=(10, 10))
        ax.imshow(disp, cmap="gray")
        ax.set_title(f"{base_name} | Accepted: {len(df_acc)} | Rejected: {len(df_rej)}")
        ax.axis("off")

        for k, row in df_acc.iterrows():
            x0, y0 = row["x_px"], row["y_px"]
            circ = plt.Circle((x0, y0), ROI_HALF_SIZE_PX, fill=False, linewidth=1.5)
            ax.add_patch(circ)
            ax.text(x0 + ROI_HALF_SIZE_PX + 2, y0, f"A{k}", fontsize=7)

        for k, row in df_rej.iterrows():
            x0, y0 = row["x_px"], row["y_px"]
            ax.plot([x0 - 5, x0 + 5], [y0 - 5, y0 + 5], linewidth=1.2)
            ax.plot([x0 - 5, x0 + 5], [y0 + 5, y0 - 5], linewidth=1.2)
            short = str(row["reasons"]).split(";")[0].strip()[:18]
            ax.text(x0 + 6, y0, f"R{k}:{short}", fontsize=6)

        fig.savefig(out_dir / f"{base_name}_qc_overlay.png", dpi=300, bbox_inches="tight")
        plt.close(fig)
    except Exception as e:
        print(f"[WARN] QC overlay failed for {base_name}: {e}")

    # Montage of accepted ROIs
    try:
        rois_acc = []
        for row in df_acc.itertuples(index=False):
            roi = extract_roi(img, row.x_px, row.y_px, ROI_HALF_SIZE_PX)
            if roi is None:
                continue
            roi_bs = np.clip(roi - (row.bg_mean if np.isfinite(row.bg_mean) else 0.0), 0, None)
            if np.max(roi_bs) > 0:
                roi_bs = roi_bs / np.max(roi_bs)
            rois_acc.append(roi_bs)

        montage = make_montage(rois_acc, ncols=10, pad=2)
        if montage is not None:
            fig, ax = plt.subplots(figsize=(12, 6))
            ax.imshow(montage, cmap="gray")
            ax.set_title("Accepted bead ROIs (background-subtracted, normalized)")
            ax.axis("off")
            fig.savefig(out_dir / f"{base_name}_qc_rois_accepted.png", dpi=300, bbox_inches="tight")
            plt.close(fig)
    except Exception as e:
        print(f"[WARN] ROI montage failed for {base_name}: {e}")

    return df_acc, df_rej, summary_lines


def main() -> None:
    in_dir = Path(INPUT_FOLDER)
    if not in_dir.exists():
        raise FileNotFoundError(f"INPUT_FOLDER not found: {in_dir.resolve()}")

    out_dir = in_dir / "PSF_analysis_outputs"
    out_dir.mkdir(exist_ok=True)

    if FILE_NAME is not None:
        paths = [in_dir / FILE_NAME]
    elif FILE_GLOB is not None:
        paths = sorted(in_dir.glob(FILE_GLOB))
        if len(paths) == 0:
            raise FileNotFoundError(f"No files matched FILE_GLOB='{FILE_GLOB}' in {in_dir}")
    else:
        raise ValueError("Set FILE_NAME or FILE_GLOB.")

    all_acc, all_rej = [], []

    for p in paths:
        if not p.exists():
            raise FileNotFoundError(f"Not found: {p}")
        img = load_image(p)
        base = p.stem

        df_acc, df_rej, summary = process_one_image(img, PIXEL_SIZE_UM, out_dir, base)
        if len(df_acc) > 0:
            df_acc.insert(0, "file", base)
            all_acc.append(df_acc)
        if len(df_rej) > 0:
            df_rej.insert(0, "file", base)
            all_rej.append(df_rej)

        print("\n".join(summary))
        print("-" * 60)

    dfA = pd.concat(all_acc, ignore_index=True) if all_acc else pd.DataFrame()
    dfR = pd.concat(all_rej, ignore_index=True) if all_rej else pd.DataFrame()

    if len(dfA) > 0:
        dfA.to_csv(out_dir / "ALL_psf_results.csv", index=False)
    if len(dfR) > 0:
        dfR.to_csv(out_dir / "ALL_psf_rejected.csv", index=False)

    if len(dfA) > 0:
        fx = dfA["fwhm_x_nm"].to_numpy()
        fy = dfA["fwhm_y_nm"].to_numpy()
        lines = [
            "OVERALL SUMMARY",
            f"Total accepted beads: {len(dfA)}",
            f"FWHM_x (nm): mean {np.nanmean(fx):.1f}, SD {np.nanstd(fx):.1f}",
            f"FWHM_y (nm): mean {np.nanmean(fy):.1f}, SD {np.nanstd(fy):.1f}",
        ]
        if DO_BEAD_SIZE_CORRECTION:
            fxc = dfA["fwhm_x_nm_corr"].to_numpy()
            fyc = dfA["fwhm_y_nm_corr"].to_numpy()
            lines += [
                f"Corrected FWHM_x (nm): mean {np.nanmean(fxc):.1f}, SD {np.nanstd(fxc):.1f}",
                f"Corrected FWHM_y (nm): mean {np.nanmean(fyc):.1f}, SD {np.nanstd(fyc):.1f}",
            ]
        (out_dir / "ALL_psf_summary.txt").write_text("\n".join(lines))
        print("\n".join(lines))

    print(f"\nDone. Outputs in: {out_dir.resolve()}")


if __name__ == "__main__":
    main()
